# 07 · EDA step 6: home ice and rest

**Question:** beyond how good the two teams are, does the setting of the game change who
wins?
- **Home ice:** home game vs. neutral site.
- **Rest:** days since each team's last game, and back-to-backs.
- **Series:** the second game against the same opponent, and who won the first.

**Why this matters for the model.** Context features are known before every game and cost
nothing to collect, but they're easy to fool yourself with. Good teams may simply be rested
more often, or host more neutral-site tournaments. So each context feature is judged **on
top of team strength**, not alone.
- If it still helps once strength is known, it goes in the model.
- If it only looked useful because it travels with strength, it stays out.

## Rules (fixed before running, same as step 5)

- **Strength model:** logistic regression on the two core step 5 features, close-game shot
  share and goal differential without empty-net goals (both season to date).
- **Context test:** for each context feature, compare the strength model with and without
  it, on folds 1–4.
  - **gain** = log loss without − log loss with.
  - **Keep it if** the mean gain is above 0 and it beats the strength model in at least 3
    of 4 folds.
- **Confirm once on fold 5.** A kept feature that doesn't help there is flagged.
- **Sizes:** each effect is reported in win-probability points for two evenly matched teams,
  with a 95% bootstrap interval, so it's readable as hockey.

In [ ]:
%run ./02_choose_k.ipynb

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

assert K == 5
REST_SQL = """
SELECT game_id, team_id, rest_days FROM features.team_games WHERE season = %s
"""  # includes non-D-I games: rest is about the schedule, not the stats
with psycopg.connect(settings.database_url) as conn:
    rest = query(conn, REST_SQL, (SEASON,))
rest_of = rest.set_index(["game_id", "team_id"])["rest_days"]

data = rows.reset_index(drop=True).copy()
y = data["home_win"].to_numpy()
splits = list(TimeSeriesSplit(n_splits=K).split(data))
SELECT, CONFIRM = splits[:4], splits[4]
STRENGTH = ["diff_close_shot_share_season", "diff_goal_diff_pg_no_en_season"]
print(f"{len(data)} games")

## Context features

- **home_ice:** 1 = home game, 0 = neutral site.
- **rest_diff:** home rest days − away rest days, each capped at 7 (a team's first game
  counts as 7).
- **home_b2b, away_b2b:** the team played the day before.
- **series_g2:** the second game against the same opponent within 2 days.
- **g1_result:** in a series' second game, +1 if the home team won the first, −1 if it
  lost, 0 for a tie or not a series game.

In [ ]:
def rest_days(game_ids, team_ids):
    r = rest_of.reindex(pd.MultiIndex.from_arrays([game_ids, team_ids])).to_numpy()
    return np.minimum(np.nan_to_num(r.astype(float), nan=7.0), 7.0)


data["home_rest"] = rest_days(data["game_id"], data["home_id"])
data["away_rest"] = rest_days(data["game_id"], data["away_id"])
data["rest_diff"] = data["home_rest"] - data["away_rest"]
data["home_b2b"] = (data["home_rest"] == 1).astype(int)
data["away_b2b"] = (data["away_rest"] == 1).astype(int)

# Previous game of the home team: same opponent within 2 days = second game of a series
tg_sorted = team_games.sort_values(["team_id", "game_date", "game_id"])
prev = tg_sorted.groupby("team_id")[["game_date", "opponent_team_id", "result"]].shift(1)
prev.columns = ["prev_date", "prev_opp", "prev_result"]
tg_prev = pd.concat([tg_sorted[["game_id", "team_id", "game_date"]], prev], axis=1)
hp = tg_prev.set_index(["game_id", "team_id"]).reindex(
    pd.MultiIndex.from_frame(data[["game_id", "home_id"]])).reset_index(drop=True)
is_g2 = (hp["prev_opp"].to_numpy() == data["away_id"].to_numpy()) & (
    (data["game_date"].to_numpy() - hp["prev_date"].to_numpy()) <= np.timedelta64(2, "D"))
data["series_g2"] = is_g2.astype(int)
data["g1_result"] = np.where(is_g2, hp["prev_result"].map({"W": 1, "L": -1, "T": 0}).fillna(0), 0)

CONTEXT = ["home_ice", "rest_diff", "home_b2b", "away_b2b", "series_g2", "g1_result"]
summary = pd.Series({
    "neutral-site games": int((data["home_ice"] == 0).sum()),
    "games with unequal rest": int((data["rest_diff"] != 0).sum()),
    "home team on a back-to-back": int(data["home_b2b"].sum()),
    "away team on a back-to-back": int(data["away_b2b"].sum()),
    "second games of a series": int(data["series_g2"].sum()),
})
summary.to_frame("games")

## 1. Raw win %

Home win % by group, with a 95% interval. This is **before** accounting for team strength.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return c - h, c + h


def raw(col, labels):
    t = data.groupby(col)["home_win"].agg(games="size", wins="sum")
    t["home win %"] = t["wins"] / t["games"]
    t["95% low"], t["95% high"] = wilson(t["wins"], t["games"])
    t.index = [labels.get(i, i) for i in t.index]
    return t


rest_bins = pd.cut(data["rest_diff"], [-8, -3, -1, 0, 2, 7], labels=["away 3+ more", "away 1-2 more", "equal", "home 1-2 more", "home 3+ more"])
data["rest_group"] = rest_bins
b2b = np.select([(data["home_b2b"] == 1) & (data["away_b2b"] == 1), data["home_b2b"] == 1, data["away_b2b"] == 1],
                ["both", "home only", "away only"], "neither")
data["b2b_group"] = b2b
data["g1_group"] = np.select([data["series_g2"] == 0, data["g1_result"] == 1, data["g1_result"] == -1],
                             ["not a 2nd game", "home won game 1", "home lost game 1"], "game 1 tied")

tables = {
    "venue": raw("home_ice", {1: "home game", 0: "neutral site"}),
    "rest": raw("rest_group", {}),
    "back-to-back": raw("b2b_group", {}),
    "series": raw("g1_group", {}),
}
pd.concat(tables).style.format({"home win %": "{:.1%}", "95% low": "{:.1%}", "95% high": "{:.1%}"})

## 2. On top of team strength (the test that decides)

In [ ]:
def fold_loss(cols, folds):
    out = []
    for train, test in folds:
        m = make_pipeline(StandardScaler(), LogisticRegression()).fit(data.loc[train, cols], y[train])
        p = m.predict_proba(data.loc[test, cols])[:, 1]
        out.append(log_loss(y[test], p, labels=[0, 1]))
    return np.array(out)


base_sel = fold_loss(STRENGTH, SELECT)
tests = {}
for c in CONTEXT:
    with_c = fold_loss(STRENGTH + [c], SELECT)
    g = base_sel - with_c
    tests[c] = {"mean gain": g.mean(), "folds beating strength": int((g > 0).sum()), "worst fold": g.min()}
tests = pd.DataFrame(tests).T
tests["kept"] = (tests["mean gain"] > 0) & (tests["folds beating strength"] >= 3)
print(f"Strength model log loss, folds 1-4: {base_sel.mean():.4f}")
tests.sort_values("mean gain", ascending=False).style.format({"mean gain": "{:+.4f}", "worst fold": "{:+.4f}"})

## 3. How big is each effect? (win-probability points for even teams)

Each effect gets **its own** logistic regression: strength plus that one feature, fit on
games before fold 5. The number is the change in home win probability for two evenly
matched teams that differ only in that feature. 95% intervals come from 500 bootstrap
resamples of the games.

**Back-to-backs are left out.** In 423 of the 432 back-to-back games, *both* teams were on
one (Friday/Saturday series). Only 9 games had just one team on a back-to-back, far too few
to measure an effect.

In [ ]:
train_all = data.iloc[: CONFIRM[1][0]]
EFFECTS = {  # feature: (from, to) values compared, label
    "home_ice": (0, 1, "home game vs neutral site"),
    "rest_diff": (0, 5, "home rested 5 more days"),
    "g1_result": (-1, 1, "home won game 1 vs lost it (2nd games only)"),
}


def effect(df, f, a, b):
    if f == "g1_result":
        df = df[df["series_g2"] == 1]
    cols = STRENGTH + [f]
    m = LogisticRegression(C=1e6, max_iter=1000).fit(df[cols], df["home_win"])
    lo = pd.DataFrame([dict.fromkeys(cols, 0.0)]); hi = lo.copy()
    lo[f], hi[f] = a, b
    return m.predict_proba(hi[cols])[0, 1] - m.predict_proba(lo[cols])[0, 1]


rng = np.random.default_rng(0)
rows_eff = []
for f, (a, b, label) in EFFECTS.items():
    point = effect(train_all, f, a, b)
    boot = [effect(train_all.sample(len(train_all), replace=True, random_state=rng.integers(1e9)), f, a, b)
            for _ in range(500)]
    rows_eff.append({"": label, "effect": point, "95% low": np.quantile(boot, 0.025),
                     "95% high": np.quantile(boot, 0.975)})
eff = pd.DataFrame(rows_eff).set_index("")
eff.style.format("{:+.1%}")

In [ ]:
# The home edge itself: how often an evenly matched home team wins, from the strength model's
# intercept (games before fold 5; 94% of them are home games, 6% neutral)
def even_home_win(df):
    m = LogisticRegression(C=1e6, max_iter=1000).fit(df[STRENGTH], df["home_win"])
    return m.predict_proba(pd.DataFrame([dict.fromkeys(STRENGTH, 0.0)]))[0, 1]


edge = even_home_win(train_all)
boot = [even_home_win(train_all.sample(len(train_all), replace=True, random_state=s)) for s in range(500)]
print(f"Evenly matched teams: home team wins {edge:.1%} "
      f"(95% interval {np.quantile(boot, 0.025):.1%} to {np.quantile(boot, 0.975):.1%})")

In [ ]:
BLUE, MUTED, GRID, INK = "#2a78d6", "#6b6a66", "#e1e0d9", "#1f1f1d"
fig, ax = plt.subplots(figsize=(7.5, 3.2))
yy = np.arange(len(eff))[::-1]
ax.hlines(yy, eff["95% low"], eff["95% high"], color=BLUE, lw=2, alpha=0.35)
ax.plot(eff["effect"], yy, "o", color=BLUE, ms=8)
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_yticks(yy, eff.index)
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_xlabel("change in home win probability, evenly matched teams (95% interval)", color=INK)
ax.set_title("Game context on top of team strength", loc="left", fontsize=11)
ax.grid(axis="x", color=GRID, lw=0.6)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Confirm once on fold 5

Only the context features kept on folds 1–4.

In [ ]:
kept = list(tests.index[tests["kept"]])
base5 = fold_loss(STRENGTH, [CONFIRM])[0]
conf = pd.DataFrame({c: {"gain on fold 5": base5 - fold_loss(STRENGTH + [c], [CONFIRM])[0]} for c in kept}).T
if len(conf):
    conf["confirmed"] = conf["gain on fold 5"] > 0
print(f"Strength model log loss on fold 5: {base5:.4f}")
conf.style.format({"gain on fold 5": "{:+.4f}"}) if len(conf) else "No context feature was kept on folds 1-4." 

## Implications for the model

**No context feature earns a place once team strength is known.** All six lost to the
strength-only model on average, and none beat it in more than 2 of 4 folds. So none was
kept, and there was nothing to confirm on fold 5.

**1. Home ice: real but small, and already covered by the intercept.**
- Evenly matched, the home team wins **53.4%** (95% interval 49–57%).
- Overall home teams win 55.7% at home and 50% at neutral sites.
- The home edge lives in the model's **intercept**, because 94% of games are home games.
- A separate home-vs-neutral feature can't be learned from 54 neutral-site games. Its
  estimate is +12 points, but with an interval from −9 to +30.
- **For the model:** no home_ice feature. Neutral-site games will get a slightly too-high
  home probability, about 3 points, from the intercept. That matters mostly in tournament
  games. Step 7 can fix it by **setting the home edge to zero at neutral sites by rule**,
  without learning it from 54 games. That's a modeling choice for step 7, not a fitted
  feature.

**2. Rest: no usable signal.**
- 758 of 957 games have both teams equally rested, because series are played on the same
  days.
- The one striking group (home rested 3+ more days: 82%) is 27 games, and the effect
  disappears in the fold test.
- **For the model:** leave rest out.

**3. Back-to-backs: can't be measured, and don't need to be.**
- In 423 of 432 back-to-back games both teams were on one, so they cancel out.
- **For the model:** leave it out. It would only matter in the rare one-sided case, and
  there's no data to estimate that.

**4. Who won game 1 of a series tells you nothing extra.**
- Raw numbers look striking: home teams that won game 1 win game 2 at 61.5%, and those
  that lost it win at 44.5%.
- But on top of strength the effect is **0.0 points** (−14 to +14). The game 1 winner was
  usually just the better team, which the season features already know.
- There's no "momentum" or "bounce-back" effect to model, consistent with step 5, where
  streak and form vs. season were weak.

**Bottom line for step 7:** the model is **team strength + intercept**. Context adds noise,
not signal. That keeps the model small, which is what about 1,000 games can support.